In [4]:
import sys
sys.path.append("../../")

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from utils.MultiScaler import MultiScaler
from sklearn.utils.validation import check_is_fitted

sns.set_theme(style="whitegrid")

In [5]:
paths = [
    "../../data/processed/X_train.csv",
    "../../data/processed/X_test.csv",
]

X_train = pd.read_csv(paths[0])
X_test = pd.read_csv(paths[1])

if list(X_train.columns) != list(X_test.columns):
    raise ValueError(
        "X_train e X_test precisam ter as mesmas colunas, na mesma ordem."
    )

print(f"Treino: {X_train.shape}")
print(f"Teste:  {X_test.shape}")
display(X_train.head())

Treino: (10648, 24)
Teste:  (4564, 24)


,GENDER,WBC,NE#,LY#,MO#,EO#,BA#,RBC,HGB,HCT,...,PLT,MPV,PCT,PDW,SD,SDTSD,TSD,FERRITTE,FOLATE,B12
0,0,6.04,3.57,1.87,0.52,0.04,0.04,4.95,11.56,38.33,...,238.1,10.07,0.24,19.52,28.80,438.60,6.566347,8.09,10.63,212.1
1,0,5.59,3.30,1.77,0.38,0.08,0.06,4.82,14.81,41.26,...,237.6,7.91,0.19,18.83,89.87,320.67,28.025696,47.39,11.69,420.8
2,0,5.96,3.27,1.87,0.62,0.16,0.04,4.65,13.90,41.30,...,188.0,10.80,0.20,11.90,47.00,332.00,14.156627,34.34,20.00,312.4
3,1,7.15,3.88,2.52,0.44,0.25,0.06,4.88,14.70,43.60,...,254.0,9.80,0.25,11.10,95.71,329.01,29.090301,55.01,7.02,149.1
4,0,10.75,6.38,3.09,1.06,0.20,0.02,4.18,12.20,34.20,...,207.0,10.10,0.21,9.30,71.99,304.99,23.604053,206.40,8.99,755.9


In [3]:
scaling_methods = {
    "StandardScaler": "standard",
    "MinMaxScaler": "minmax",
    "RobustScaler": "robust",
    "QuantileTransformer": "quantile",
}

scaled_results = {}

for method_name, method in scaling_methods.items():
    transformer = MultiScaler(
        method=method,
        skew_threshold=1.0,
        apply_log=True,
        random_state=1822,
    )

    X_train_transformed = transformer.fit_transform(X_train)
    X_test_transformed = transformer.transform(X_test)

    scaled_results[method_name] = {
        "train": X_train_transformed,
        "test": X_test_transformed,
        "transformer": transformer,
    }

print("Técnicas avaliadas:", list(scaled_results))

print(
    "Variáveis com Log1p:",
    scaled_results["StandardScaler"]["transformer"].cols_para_log_,
)

Técnicas avaliadas: ['StandardScaler', 'MinMaxScaler', 'RobustScaler', 'QuantileTransformer']
Variáveis com Log1p: ['WBC', 'NE#', 'LY#', 'MO#', 'EO#', 'BA#', 'RDW', 'PLT', 'PCT', 'SD', 'TSD', 'FERRITTE', 'FOLATE', 'B12']


In [4]:
reference_transformer = scaled_results["StandardScaler"]["transformer"]
numeric_cols = reference_transformer.num_cols_
preserved_cols = [col for col in X_train.columns if col not in numeric_cols]

display(pd.DataFrame({
    "Variável": X_train.columns,
    "Tipo original": X_train.dtypes.astype(str).values,
    "Tratamento": [
        "Log1p + escalonamento" if col in reference_transformer.cols_para_log_
        else "Escalonamento" if col in numeric_cols and col != "GENDER"
        else "Preservada (passthrough)"
        for col in X_train.columns
    ],
}))

,Variável,Tipo original,Tratamento
0,GENDER,int64,Preservada (passthrough)
1,WBC,float64,Log1p + escalonamento
2,NE#,float64,Log1p + escalonamento
3,LY#,float64,Log1p + escalonamento
4,MO#,float64,Log1p + escalonamento
5,EO#,float64,Log1p + escalonamento
6,BA#,float64,Log1p + escalonamento
7,RBC,float64,Escalonamento
8,HGB,float64,Escalonamento
9,HCT,float64,Escalonamento


In [5]:
cols_para_ignorar = {"GENDER"}
opcoes_vars = [col for col in numeric_cols if col not in cols_para_ignorar]

datasets_para_comparar = {"Sem escalonamento": X_train}

datasets_para_comparar.update({
    method_name: result["train"] 
    for method_name, result in scaled_results.items()
})

comparison_rows = []

for method_name, data in datasets_para_comparar.items():
    for col in opcoes_vars:
        values = data[col]
        comparison_rows.append({
            "Técnica": method_name,
            "Variável": col,
            "Média": values.mean(),
            "Desvio padrão": values.std(),
            "Mínimo": values.min(),
            "Q1": values.quantile(0.25),
            "Mediana": values.median(),
            "Q3": values.quantile(0.75),
            "Máximo": values.max(),
            "IQR": values.quantile(0.75) - values.quantile(0.25),
            "Assimetria": values.skew(),
        })

df_scaling_comparison = pd.DataFrame(comparison_rows)

formato_colunas = {
    "Média": "{:.3f}",
    "Desvio padrão": "{:.3f}",
    "Mínimo": "{:.3f}",
    "Q1": "{:.3f}",
    "Mediana": "{:.3f}",
    "Q3": "{:.3f}",
    "Máximo": "{:.3f}",
    "IQR": "{:.3f}",
    "Assimetria": "{:.3f}",
}


def exibir_tabela(tecnica_selecionada):
    if tecnica_selecionada == "Todos":
        df_filtrado = df_scaling_comparison
    else:
        df_filtrado = df_scaling_comparison[
            df_scaling_comparison["Técnica"] == tecnica_selecionada
        ]

    display(df_filtrado.style.format(formato_colunas))


opcoes = ["Todos"] + list(datasets_para_comparar.keys())

seletor = widgets.Dropdown(
    options=opcoes,
    value="Todos",
    description="Técnica:",
    style={"description_width": "initial"},
)

widgets.interact(exibir_tabela, tecnica_selecionada=seletor);

interactive(children=(Dropdown(description='Técnica:', options=('Todos', 'Sem escalonamento', 'StandardScaler'…

In [6]:
datasets_para_resumo = {"Sem escalonamento": X_train}
datasets_para_resumo.update({
    method_name: result["train"]
    for method_name, result in scaled_results.items()
})

summary_rows = []

for method_name, df_atual in datasets_para_resumo.items():
    data = df_atual[numeric_cols]

    feature_means = data.mean()
    feature_stds = data.std()
    feature_skews = data.skew()

    summary_rows.append({
        "Técnica": method_name,
        "Média do |valor médio|": feature_means.abs().mean(),
        "Média do desvio padrão": feature_stds.mean(),
        "Média do |skewness|": feature_skews.abs().mean(),
        "Maior |skewness|": feature_skews.abs().max(),
        "Média da amplitude": (data.max() - data.min()).mean(),
    })

df_scaling_summary = pd.DataFrame(summary_rows).set_index("Técnica")
display(df_scaling_summary.style.format("{:.3f}"))

,Média do |valor médio|,Média do desvio padrão,Média do |skewness|,Maior |skewness|,Média da amplitude
Técnica,,,,,
Sem escalonamento,61.905,50.546,11.514,85.699,2704.159
StandardScaler,0.000,1.000,1.663,22.280,12.528
MinMaxScaler,0.372,0.114,1.663,22.280,1.000
RobustScaler,0.095,0.845,1.663,22.280,12.307
QuantileTransformer,0.493,0.296,0.035,0.771,1.000


In [7]:
def plot_scaling_comparison(col):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.ravel()

    for ax, (method_name, result) in zip(axes, scaled_results.items()):
        sns.histplot(
            result["train"][col],
            kde=True,
            stat="density",
            bins=30,
            alpha=0.45,
            ax=ax,
        )
        ax.set_title(method_name)
        ax.set_xlabel("Valor transformado")
        ax.set_ylabel("Densidade")

    fig.suptitle(f"Distribuição de {col} por técnica", fontsize=15)
    plt.tight_layout()
    plt.show()

    box_data = pd.DataFrame({
        method_name: result["train"][col].reset_index(drop=True)
        for method_name, result in scaled_results.items()
    })

    plt.figure(figsize=(13, 5))
    sns.boxplot(data=box_data)
    plt.title(f"Boxplots de {col} por técnica")
    plt.xlabel("Técnica")
    plt.ylabel("Valor transformado")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()


cols_para_ignorar = {"GENDER"}
opcoes_vars = [col for col in numeric_cols if col not in cols_para_ignorar]

scaling_selector = widgets.Dropdown(
    options=opcoes_vars,
    description="Variável:",
)
scaling_output = widgets.interactive_output(
    plot_scaling_comparison,
    {"col": scaling_selector},
)
display(scaling_selector, scaling_output)

Dropdown(description='Variável:', options=('WBC', 'NE#', 'LY#', 'MO#', 'EO#', 'BA#', 'RBC', 'HGB', 'HCT', 'MCV…

Output()

In [8]:
ordem_colunas = [
    "Sem escalonamento",
    "StandardScaler",
    "MinMaxScaler",
    "RobustScaler",
    "QuantileTransformer",
]

vars_ignorar = ["GENDER"]

skew_comparison = (
    df_scaling_comparison
    .pivot(index="Variável", columns="Técnica", values="Assimetria")
    .reindex(columns=ordem_colunas)
    .drop(index=vars_ignorar, errors="ignore")
)

skew_plot_base = skew_comparison.iloc[::-1]

checkboxes = {
    tec: widgets.Checkbox(
        value=True,
        description=tec,
        indent=False,
        layout=widgets.Layout(width="auto", margin="0 15px 0 0")
    )
    for tec in skew_plot_base.columns
}

painel_checks = widgets.HBox(
    list(checkboxes.values()),
    layout=widgets.Layout(flex_flow="row wrap", margin="0 0 10px 0")
)

output_grafico = widgets.Output()


def atualizar_grafico(*args):
    selecionadas = [tec for tec, cb in checkboxes.items() if cb.value]

    with output_grafico:
        output_grafico.clear_output(wait=True)

        if not selecionadas:
            print("⚠️ Selecione pelo menos uma técnica acima para visualizar o gráfico.")
            return

        df_filtrado = skew_plot_base[selecionadas]
        n_vars = len(df_filtrado)
        altura_figura = max(5.5, n_vars * (0.28 + 0.06 * len(selecionadas)))

        fig, ax = plt.subplots(figsize=(11, altura_figura))
        df_filtrado.plot(kind="barh", ax=ax, width=0.78)

        ax.axvline(0, color="black", linewidth=1.0)
        ax.axvline(
            1.0,
            color="red",
            linestyle="--",
            linewidth=0.9,
            alpha=0.7,
            label="Limite (±1.0)"
        )
        ax.axvline(-1.0, color="red", linestyle="--", linewidth=0.9, alpha=0.7)
        ax.axvspan(
            -1.0, 1.0, color="green", alpha=0.05, label="Zona ideal (-1 a 1)"
        )

        ax.set_title(
            "Assimetria por variável e técnica (ordenadas por assimetria original)",
            fontsize=12,
            fontweight="bold",
            pad=55
        )

        ax.legend(
            bbox_to_anchor=(0.5, 1.01),
            loc="lower center",
            ncol=min(4, len(selecionadas) + 2),
            frameon=True,
            fontsize=9
        )

        ax.set_xlabel("Assimetria (skewness)")
        ax.set_ylabel("")
        ax.grid(axis="x", linestyle=":", alpha=0.6)
        ax.grid(axis="y", visible=False)

        plt.tight_layout()
        plt.show()


for cb in checkboxes.values():
    cb.observe(atualizar_grafico, names="value")

display(widgets.VBox([
    widgets.HTML("<b>Selecione as técnicas para comparar:</b>"),
    painel_checks,
    output_grafico
]))

atualizar_grafico()

In [9]:
ordem_desc = (
    skew_comparison["Sem escalonamento"]
    .abs()
    .sort_values(ascending=False).index
)

skew_comparison = skew_comparison.loc[ordem_desc]


def destacar_menor_assimetria(linha):
    valores_abs = linha.abs()
    eh_minimo = np.isclose(valores_abs, valores_abs.min(), rtol=1e-5, atol=1e-8)
    return ["font-weight: bold;" if v else "" for v in eh_minimo]


display(
    skew_comparison.style
    .format("{:.3f}")
    .apply(destacar_menor_assimetria, axis=1)
)

Técnica,Sem escalonamento,StandardScaler,MinMaxScaler,RobustScaler,QuantileTransformer
Variável,,,,,
BA#,85.699,22.280,22.280,22.280,0.003
B12,39.492,0.704,0.704,0.704,-0.006
FERRITTE,32.680,0.453,0.453,0.453,-0.001
LY#,24.531,1.100,1.100,1.100,-0.002
MO#,22.077,2.032,2.032,2.032,0.007
WBC,19.453,0.607,0.607,0.607,-0.004
NE#,19.220,0.589,0.589,0.589,-0.002
EO#,18.351,3.757,3.757,3.757,-0.004
PCT,1.794,1.026,1.026,1.026,0.005
